# 3 · Comparação de pré-processamento clássico

Comparação exploratória na **validação**, sem usar o teste para ajustar a técnica. Cada combinação usa a mesma amostra de imagens e caixas: RGB/cinza, saturação HSV ou distância CIELAB; com imagem original, redução de ruído ou correção de iluminação. Os métodos usam limiarização, abertura/fechamento, contornos e filtros geométricos. O detector YOLO e seu teste completo estão no notebook 1.

In [ ]:
from pathlib import Path
import os,time
import cv2,numpy as np,pandas as pd
def read_image(path):
    try: return cv2.imdecode(np.fromfile(str(path),dtype=np.uint8),cv2.IMREAD_COLOR)
    except (OSError,cv2.error): return None
OUTPUT=Path(os.environ.get("RESISTOR_OUTPUT","../outputs/proposta")).expanduser().resolve()
PREPARED=OUTPUT/"samu-agrupado"
manifest=pd.read_csv(PREPARED/"manifesto.csv")
assert not manifest.empty
# Subconjunto determinístico da validação para uma comparação rápida e reproduzível.
limit=int(os.environ.get("RESISTOR_PREPROCESS_SAMPLE","200"))
samples=manifest[manifest.split=="val"].sample(n=min(limit,(manifest.split=="val").sum()),random_state=42)
print("Amostras de validação:",len(samples))


In [ ]:
def preprocess(image, option):
    if option=="original": return image
    if option=="ruido": return cv2.GaussianBlur(image,(5,5),0)
    lab=cv2.cvtColor(image,cv2.COLOR_BGR2LAB)
    l,a,b=cv2.split(lab)
    l=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(l)
    return cv2.cvtColor(cv2.merge((l,a,b)),cv2.COLOR_LAB2BGR)
def classic(image,space):
    if space=="RGB":
        channel=cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)
    elif space=="HSV":
        channel=cv2.cvtColor(image,cv2.COLOR_BGR2HSV)[:,:,1]
    else:
        lab=cv2.cvtColor(image,cv2.COLOR_BGR2LAB).astype(np.float32)
        border=np.concatenate((lab[0,:,:],lab[-1,:,:],lab[:,0,:],lab[:,-1,:]))
        channel=np.uint8(np.clip(np.linalg.norm(lab-np.median(border,axis=0),axis=2),0,255))
    _,mask=cv2.threshold(channel,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    border=np.concatenate((mask[0,:],mask[-1,:],mask[:,0],mask[:,-1]))
    if np.median(border)>127: mask=cv2.bitwise_not(mask)
    kernel=np.ones((3,3),np.uint8)
    mask=cv2.morphologyEx(mask,cv2.MORPH_OPEN,kernel)
    mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,kernel,iterations=2)
    contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    out=[]
    for c in contours:
        area=cv2.contourArea(c)
        if not 350<=area<=image.shape[0]*image.shape[1]*.8: continue
        _,(w,h),_=cv2.minAreaRect(c)
        short,long=sorted((w,h))
        if short<8 or not 1.8<=long/short<=12: continue
        x,y,w,h=cv2.boundingRect(c); out.append((x,y,x+w,y+h))
    return out

def iou(a,b):
    ix=max(0,min(a[2],b[2])-max(a[0],b[0]))*max(0,min(a[3],b[3])-max(a[1],b[1]))
    return ix/max(1e-9,(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-ix)
def match(gt,pred):
    pairs=sorted(((iou(a,b),i,j) for i,a in enumerate(gt) for j,b in enumerate(pred)),reverse=True)
    a,b=set(),set()
    for v,i,j in pairs:
        if v<.5: break
        if i not in a and j not in b: a.add(i);b.add(j)
    return len(a),len(pred)-len(b),len(gt)-len(a)


In [ ]:
records=[]
for row in samples.itertuples(index=False):
    path=next((PREPARED/"val/images").glob(row.id+".*"))
    image=read_image(path); assert image is not None
    h,w=image.shape[:2]; truth=[]
    for line in (PREPARED/"val/labels"/(row.id+".txt")).read_text().splitlines():
        _,x,y,bw,bh=map(float,line.split())
        truth.append(((x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h))
    for option in ("original","ruido","iluminacao"):
        for space in ("RGB","HSV","CIELAB"):
            start=time.perf_counter(); boxes=classic(preprocess(image,option),space)
            elapsed=(time.perf_counter()-start)*1000
            tp,fp,fn=match(truth,boxes)
            records.append({"id":row.id,"preprocessamento":option,"espaco":space,"esperado":len(truth),"detectado":len(boxes),"tp":tp,"fp":fp,"fn":fn,"ms":elapsed})
result=pd.DataFrame(records)
result.to_csv(OUTPUT/"preprocessamento_por_imagem.csv",index=False)
def metrics(g):
    tp,fp,fn=g.tp.sum(),g.fp.sum(),g.fn.sum()
    return pd.Series({"imagens":len(g),"contagem_exata":(g.esperado==g.detectado).mean(),"erro_absoluto_medio":(g.esperado-g.detectado).abs().mean(),"precisao_iou50":tp/(tp+fp) if tp+fp else 0.,"recall_iou50":tp/(tp+fn) if tp+fn else 0.,"tempo_medio_ms":g.ms.mean()})
summary=result.groupby(["preprocessamento","espaco"]).apply(metrics,include_groups=False)
summary.to_csv(OUTPUT/"preprocessamento_resumo.csv")
display(summary)


Estes resultados são de validação exploratória, não representam mAP nem autorizam comparar tempos com GPU diretamente. Os filtros são fixos e, por isso, podem falhar em imagens cujo fundo seja complexo.